# 🐱🐶 Dog vs Cat Image Classification Using CNN

## Project Overview

This project builds a **Convolutional Neural Network (CNN)** to classify images into two categories:

* 🐱 **Cat**
* 🐶 **Dog**

The model is trained using a labeled image dataset and learns visual patterns that help distinguish between cats and dogs.

The complete workflow includes:

* Loading and exploring the image dataset
* Preprocessing and normalizing images
* Applying data augmentation
* Building a Convolutional Neural Network (CNN)
* Training and evaluating the model
* Visualizing training and validation performance
* Analyzing predictions using a confusion matrix and classification report
* Predicting random images
* Saving and loading the trained model


## 1. Import Libraries

The following libraries are used for image processing, dataset handling, CNN model building, visualization, and model evaluation.


In [ ]:
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import load_img, img_to_array

from sklearn.metrics import classification_report, confusion_matrix

## 2. Load the Dataset

The dataset is loaded from the project's `data/raw/` directory.

The images are organized into two classes:

* 🐱 **Cats**
* 🐶 **Dogs**

TensorFlow's `image_dataset_from_directory()` automatically reads the images and assigns labels based on the folder names.


In [ ]:
train_dataset = keras.utils.image_dataset_from_directory(
    directory="../data/raw/training_set",
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(256, 256),
    shuffle=True,
    seed=42
)

test_dataset = keras.utils.image_dataset_from_directory(
    directory="../data/raw/test_set",
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(256, 256),
    shuffle=False
)

# verify the class names
print("Classes:", train_dataset.class_names)

## 3. Explore and Visualize the Dataset

Before training the CNN, it is useful to inspect some images from the dataset.

This helps us verify that:

* The images are loaded correctly.
* Both classes are present.
* The labels match the correct images.


In [ ]:
plt.figure(figsize=(12, 8))

for images, labels in train_dataset.take(1):
    
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        
        class_name = train_dataset.class_names[labels[i]]
        
        plt.title(
            class_name.capitalize(),
            fontsize=13,
            fontweight="bold"
        )
        
        plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Inspect Image Shape and Labels

Before preprocessing the images, we inspect one batch to understand how the dataset is structured.

This allows us to check:

* Image batch shape
* Label batch shape
* Image data type
* Sample labels


In [ ]:
for images, labels in train_dataset.take(1):
    print("Images batch shape:", images.shape)
    print("Labels batch shape:", labels.shape)

    print("\nImage data type:", images.dtype)
    print("Labels:", labels[:10].numpy())

## 5. Check Pixel Values and Normalize Images

Digital images store pixel values between **0 and 255**.

Neural networks generally train more efficiently when the input values are scaled to a smaller range. Therefore, we normalize the pixel values from:

**0–255 → 0–1**

This is done by dividing each pixel value by `255`.


In [ ]:
# Inspecting 
for images, labels in train_dataset.take(1):
    print("Minimum pixel value:", images.numpy().min())
    print("Maximum pixel value:", images.numpy().max())


# Normalize
normalization_layer = layers.Rescaling(1.0 / 255)

train_dataset = train_dataset.map(
    lambda images, labels: (
        normalization_layer(images),
        labels
    )
)

test_dataset = test_dataset.map(
    lambda images, labels: (
        normalization_layer(images),
        labels
    )
)


# verify
for images, labels in train_dataset.take(1):
    print("Minimum pixel value:", images.numpy().min())
    print("Maximum pixel value:", images.numpy().max())

## 6. Dataset Optimization

During training, the model repeatedly reads batches of images from the dataset. We can improve the data pipeline using **cache** and **prefetch**.

* **Cache:** Stores processed data to reduce repeated loading.
* **Prefetch:** Prepares the next batch while the model is processing the current batch.

This helps improve the efficiency of the training pipeline.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(
    buffer_size=AUTOTUNE
)

test_dataset = test_dataset.cache().prefetch(
    buffer_size=AUTOTUNE
)

## 7. Data Augmentation

Data augmentation artificially creates variations of training images to help the model learn more robust features.

Instead of permanently creating and saving new images, the transformations are applied dynamically during training.

In this project, we use:

* **Random Flip** → Flips images horizontally.
* **Random Rotation** → Slightly rotates images.
* **Random Zoom** → Randomly zooms into images.

These variations help reduce overfitting and improve the model's ability to handle different image orientations and appearances.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

# Visualize the augmented images
plt.figure(figsize=(10, 8))

for images, labels in train_dataset.take(1):
    
    image = images[0]
    
    for i in range(6):
        augmented_image = data_augmentation(
            tf.expand_dims(image, 0)
        )
        
        plt.subplot(2, 3, i + 1)
        plt.imshow(augmented_image[0])
        plt.axis("off")

plt.tight_layout()
plt.show()

## 8. Build the CNN Model

A **Convolutional Neural Network (CNN)** is used to automatically learn visual features from images.

The model consists of three main convolution blocks. Each block uses:

* **Conv2D** to detect image features such as edges, shapes, and patterns.
* **MaxPooling2D** to reduce the spatial size of the feature maps.
* **Flatten** to convert the extracted feature maps into a single vector.
* **Dense layers** to learn patterns and perform the final classification.
* **Dropout** to reduce overfitting.

The final layer uses the **Sigmoid activation function** because this is a binary classification problem: **Cat or Dog**.


In [ ]:
model = keras.Sequential([
    
    # Data Augmentation
    data_augmentation,
    
    # First Convolution Block
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    
    # Second Convolution Block
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    
    # Third Convolution Block
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    
    # Convert feature maps into a single vector
    layers.Flatten(),
    
    # Fully Connected Layer
    layers.Dense(128, activation="relu"),
    
    # Reduce Overfitting
    layers.Dropout(0.5),
    
    # Output Layer
    layers.Dense(1, activation="sigmoid")
])

# Checking the model Architecture

In [ ]:
model.build(input_shape=(None, 256, 256, 3))

model.summary()

## 9. Compile the Model

Before training, the model needs to be configured with:

* **Optimizer** — Controls how the model updates its weights during learning.
* **Loss Function** — Measures how far the predictions are from the actual labels.
* **Accuracy** — Tracks how many predictions are correct during training.

Since this is a binary classification problem (**Cat vs Dog**), we use **Binary Crossentropy** as the loss function and **Sigmoid** in the output layer.


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## 10. Train the Model

The CNN is trained using the training dataset for multiple **epochs**.

During each epoch, the model processes the complete training dataset, calculates the loss, updates its weights, and evaluates its performance on the validation dataset.

The training history is stored in `dog_cat_model`, which allows us to visualize the training and validation accuracy and loss later.


In [ ]:
dog_cat_model = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10
)

## 11. Visualize Model Performance

After training, we visualize the model's performance across all epochs.

The graphs help us compare:

* **Training vs Validation Accuracy**
* **Training vs Validation Loss**

These visualizations help us understand how well the model learned and whether signs of overfitting or underfitting are present.


## Training vs Validation Accuracy

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    dog_cat_model.history["accuracy"],
    label="Training Accuracy",
    color="royalblue",
    linewidth=2.5,
    marker="o",
    markersize=6
)

plt.plot(
    dog_cat_model.history["val_accuracy"],
    label="Validation Accuracy",
    color="orange",
    linewidth=2.5,
    marker="o",
    markersize=6
)

plt.xlabel("Epoch", fontsize=15, fontweight="bold")
plt.ylabel("Accuracy", fontsize=15, fontweight="bold")

plt.title(
    "Training vs Validation Accuracy",
    fontsize=19,
    fontweight="bold",
    pad=15
)

plt.xticks(
    ticks=range(10),
    labels=range(1, 11),
    fontsize=13
)

plt.yticks(fontsize=13)

plt.legend(
    fontsize=13,
    loc="lower right"
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.5
)

plt.tight_layout()
plt.show()

# Training vs Validation Loss

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    dog_cat_model.history["loss"],
    label="Training Loss",
    color="darkviolet",
    linewidth=2.5,
    marker="o",
    markersize=6
)

plt.plot(
    dog_cat_model.history["val_loss"],
    label="Validation Loss",
    color="plum",
    linewidth=2.5,
    marker="o",
    markersize=6
)

plt.xlabel("Epoch", fontsize=15, fontweight="bold")
plt.ylabel("Loss", fontsize=15, fontweight="bold")

plt.title(
    "Training vs Validation Loss",
    fontsize=19,
    fontweight="bold",
    pad=15
)

plt.xticks(
    ticks=range(10),
    labels=range(1, 11),
    fontsize=13
)

plt.yticks(fontsize=13)

plt.legend(
    fontsize=13,
    loc="upper right"
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.5
)

plt.tight_layout()
plt.show()

## 12. Evaluate the Model

After training the CNN, we evaluate the model on the test dataset to measure how well it performs on unseen images.

The evaluation provides two important metrics:

* **Test Loss** → Measures the prediction error of the model.
* **Test Accuracy** → Measures the percentage of test images classified correctly.

The test dataset contains images that were not used to update the model's weights during training, so it gives us a better indication of how well the CNN generalizes to new images.


In [ ]:
test_loss, test_accuracy = model.evaluate(test_dataset)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)
print("Test Accuracy (%):", test_accuracy * 100)

## 13. Generate the Confusion Matrix

A confusion matrix helps us understand how the CNN classifies the test images.

Instead of showing only the overall accuracy, it shows:

- **Actual Cat → Predicted Cat**
- **Actual Cat → Predicted Dog**
- **Actual Dog → Predicted Cat**
- **Actual Dog → Predicted Dog**

This allows us to identify the types of mistakes made by the model.

In [ ]:
from sklearn.metrics import confusion_matrix

y_true = []
y_pred = []

for images, labels in test_dataset:
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend((predictions > 0.5).astype(int).flatten())

cm = confusion_matrix(y_true, y_pred)

print(cm)

## 14. Visualize the Confusion Matrix

The confusion matrix is visualized as a heatmap-like matrix so that we can easily understand the model's correct and incorrect predictions.

The diagonal values represent correct predictions, while the off-diagonal values represent incorrect predictions.

In [ ]:
plt.figure(figsize=(8, 6))

plt.imshow(cm)

plt.title(
    "Confusion Matrix",
    fontsize=19,
    fontweight="bold",
    pad=15
)

plt.xlabel(
    "Predicted Label",
    fontsize=15,
    fontweight="bold"
)

plt.ylabel(
    "True Label",
    fontsize=15,
    fontweight="bold"
)

plt.xticks(
    ticks=[0, 1],
    labels=["Cat", "Dog"],
    fontsize=13
)

plt.yticks(
    ticks=[0, 1],
    labels=["Cat", "Dog"],
    fontsize=13
)

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            fontsize=16,
            fontweight="bold"
        )

plt.colorbar()

plt.tight_layout()
plt.show()

## 15. Understanding the Confusion Matrix

A confusion matrix shows us **how the model's predictions compare with the actual classes**.

For our Dog vs Cat classification:

|                | Predicted Cat | Predicted Dog |
| -------------- | ------------: | ------------: |
| **Actual Cat** |   Correct Cat |     Cat → Dog |
| **Actual Dog** |     Dog → Cat |   Correct Dog |

### How to read it

```text
                    Predicted
                  Cat       Dog
               ┌────────┬────────┐
Actual Cat     │   ✓    │   ✗    │
               ├────────┼────────┤
Actual Dog     │   ✗    │   ✓    │
               └────────┴────────┘

## 16. Classification Report

The classification report provides detailed information about how well the CNN performs for each class.

It gives us four important metrics:

- **Precision:** How often the model is correct when it predicts a class.
- **Recall:** How many actual samples of a class the model correctly identifies.
- **F1-score:** A combined measure of precision and recall.
- **Support:** The number of actual samples belonging to each class.

### Our Model

| Class | Precision | Recall | F1-score | Support |
| ----- | --------: | -----: | -------: | ------: |
| Cat   |      0.73 |   0.84 |     0.78 |    1011 |
| Dog   |      0.81 |   0.69 |     0.75 |    1012 |

Overall, the model achieved **77% accuracy** on **2,023 test images**.

The model has **higher Cat recall (84%)**, meaning it finds more actual cats, while it has **higher Dog precision (81%)**, meaning its Dog predictions are more reliable.

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Cat", "Dog"]
    )
)

## 17. Test the Model with an Image

After evaluating the model on the complete test dataset, we can test the trained CNN using a single image.

In this step, we will:

1. Select a random image from the test dataset.
2. Resize the image to **256 × 256**.
3. Normalize the image pixels.
4. Add a batch dimension.
5. Pass the image to the trained CNN.
6. Display the model's prediction.

In [ ]:
import random
from pathlib import Path
from tensorflow.keras.utils import load_img, img_to_array

# Test dataset path
test_dir = Path("../data/raw/test_set")

# Get all images
image_paths = (
    list(test_dir.glob("cats/*")) +
    list(test_dir.glob("dogs/*"))
)

# Select a random image
image_path = random.choice(image_paths)

print("Selected image:", image_path)

## 18. Display the Selected Image

The selected test image is displayed before prediction so that we can visually compare the actual image with the CNN's prediction.

In [ ]:
import matplotlib.pyplot as plt

# Load image for display
display_image = load_img(image_path)

plt.figure(figsize=(6, 6))

plt.imshow(display_image)

plt.title(
    "Selected Test Image",
    fontsize=18,
    fontweight="bold"
)

plt.axis("off")

plt.tight_layout()
plt.show()

## 19. Preprocess the Image

The selected image must be preprocessed in the same way as the images used during model training.

The image is:

- Resized to 256 × 256 pixels.
- Converted into a numerical array.
- Normalized so that pixel values are between 0 and 1.
- Expanded to include the batch dimension required by the CNN.

In [ ]:
# Load and resize image
image = load_img(
    image_path,
    target_size=(256, 256)
)

# Convert image to NumPy array
image_array = img_to_array(image)

# Normalize pixel values
image_array = image_array / 255.0

# Add batch dimension
image_array = np.expand_dims(image_array, axis=0)

print("Image shape:", image_array.shape)

## 20. Make the Prediction

The preprocessed image is passed to the trained CNN.

Because this is a binary classification problem, the model produces a probability value between 0 and 1.

- A value closer to **0** represents Cat.
- A value closer to **1** represents Dog.

A threshold of **0.5** is used to determine the final predicted class.

In [ ]:
# Make prediction
prediction = model.predict(
    image_array,
    verbose=0
)

# Extract prediction probability
probability = prediction[0][0]

# Determine predicted class
if probability >= 0.5:
    predicted_class = "Dog"
else:
    predicted_class = "Cat"

print("Prediction probability:", probability)
print("Predicted class:", predicted_class)

## 21. Compare Actual and Predicted Class

The actual class of the selected image can be obtained from its parent directory.

We compare the actual class with the CNN's prediction to determine whether the model correctly classified the image.

In [ ]:
# Get actual class from folder name
actual_class = image_path.parent.name.capitalize()

print("Actual class:", actual_class)
print("Predicted class:", predicted_class)

if actual_class == predicted_class:
    print("Result: Correct Prediction")
else:
    print("Result: Incorrect Prediction")

## 22. Visualize the Final Prediction

The selected image is displayed together with:

- Actual class
- Predicted class
- Prediction probability
- Whether the prediction was correct

This provides an intuitive way to understand how the trained CNN performs on an individual unseen image.

In [ ]:
plt.figure(figsize=(7, 7))

plt.imshow(display_image)

plt.title(
    f"Actual: {actual_class} | Predicted: {predicted_class}\n"
    f"Probability: {probability:.2%}",
    fontsize=16,
    fontweight="bold"
)

plt.axis("off")

plt.tight_layout()
plt.show()

## 23. Save the Trained Model

After training and evaluating the CNN, we save the trained model so that it can be reused later without training it again.

Saving the model preserves:

- The model architecture
- The learned weights
- The optimizer configuration
- The training state

The saved model can later be loaded and used to make predictions on new images.

In [ ]:
# Save the trained model
model.save("../models/cat_dog_cnn.keras")

print("Model saved successfully!")

## 24. Load the Saved Model

To verify that the model was saved correctly, we load the `.keras` file back into memory.

This allows us to use the trained CNN later without running the training process again.

In [ ]:
from tensorflow.keras.models import load_model

# Load the saved model
saved_model = load_model("../models/cat_dog_cnn.keras")

print("Saved model loaded successfully!")

## 25. Verify the Loaded Model

After loading the saved model, we verify that it can make predictions.

We use the same test image and preprocessing pipeline to ensure that the saved model behaves correctly.

In [ ]:
# Make prediction using the loaded model
loaded_prediction = saved_model.predict(
    image_array,
    verbose=0
)

loaded_probability = loaded_prediction[0][0]

if loaded_probability >= 0.5:
    loaded_class = "Dog"
else:
    loaded_class = "Cat"

print("Prediction probability:", loaded_probability)
print("Predicted class:", loaded_class)

## 26. Final Model Summary

The Cat vs Dog CNN classification project has now completed the complete machine learning workflow.

### Workflow

1. Dataset preparation
2. Image preprocessing
3. CNN architecture creation
4. Model compilation
5. Model training
6. Model evaluation
7. Confusion matrix generation
8. Classification report
9. Single-image prediction
10. Model saving
11. Saved-model loading and verification

The trained CNN can now be reused to classify new images as either **Cat** or **Dog** without retraining the model.